# COMMONWEAVE Zero — Validación · Performance · Veredicto de eficiencia

Notebook **autónomo**: si no encuentra el prototipo en el disco, lo clona desde GitHub.

1. **Validación** — equivalencia semántica adaptador vs Python puro  
2. **Performance** — latencia mediana / p99 (pure · manual · generated · IPC)  
3. **Veredicto de eficiencia** — corrección + esfuerzo + rendimiento  

```bash
pip install pyyaml cryptography
# gcc y make deben estar disponibles para compilar el .so
```

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import statistics
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = "https://github.com/dcpracmatic-prog/verificacion_formal.git"
SUBDIR = "upstream/commonweave_zero"

def find_root() -> Path | None:
    candidates = [
        Path(".").resolve(),
        Path("..").resolve(),
        Path("../upstream/commonweave_zero").resolve(),
        Path("upstream/commonweave_zero").resolve(),
        Path("/content/verificacion_formal") / SUBDIR,
        Path.cwd() / "verificacion_formal" / SUBDIR,
        Path("/tmp/verificacion_formal") / SUBDIR,
    ]
    for c in candidates:
        if (c / "run_zero.py").exists() or (c / "contracts" / "matmul.yaml").exists():
            return c
    return None

ROOT = find_root()

if ROOT is None:
    print("Prototipo no encontrado en disco. Clonando desde GitHub…")
    clone_parent = Path("/content") if Path("/content").is_dir() else Path("/tmp")
    clone_dir = clone_parent / "verificacion_formal"
    if clone_dir.exists():
        shutil.rmtree(clone_dir)
    r = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError("No se pudo clonar el repositorio")
    ROOT = clone_dir / SUBDIR
    if not ROOT.exists():
        raise FileNotFoundError(f"Tras clonar no existe {ROOT}")
    print(f"Clonado en {clone_dir}")

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"ROOT = {ROOT}")
print(f"Python {sys.version.split()[0]}")
print("Contenido:", sorted(p.name for p in ROOT.iterdir())[:15])

## 0. Dependencias, compilar nativo y regenerar adaptador

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "cryptography"], check=False)

def sh(cmd, cwd=None):
    cwd = cwd or ROOT
    print("+", " ".join(str(x) for x in cmd))
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(f"comando falló ({r.returncode}): {cmd}")
    if r.stdout.strip():
        print(r.stdout.strip()[:600])
    return r

sh(["make"], cwd=ROOT / "c_src")
(ROOT / "generated").mkdir(exist_ok=True)
(ROOT / "report").mkdir(exist_ok=True)

sh([
    sys.executable, str(ROOT / "generator" / "generate_adapter.py"),
    "--contract", str(ROOT / "contracts" / "matmul.yaml"),
    "--lib", str(ROOT / "c_src" / "libmatmul.so"),
    "--out", str(ROOT / "generated" / "matmul_adapter.py"),
])
print("✓ nativo + adaptador listos")

## 1. Validación diferencial

In [ ]:
diff_out = ROOT / "report" / "differential_nb.json"

sh([
    sys.executable, str(ROOT / "verifier" / "differential.py"),
    "--trials", "80",
    "--seed", "20261019",
    "--out", str(diff_out),
])

diff = json.loads(diff_out.read_text())
print("\n=== RESULTADO VALIDACIÓN ===")
print(f"  veredicto   : {diff['verdict']}")
print(f"  pass_rate   : {diff['pass_rate']:.1%}  ({diff['passed']}/{diff['trials']})")
print(f"  edge cases  : {diff['edge_passed']}/{diff['edge_total']}")
assert diff["verdict"] == "PASS", "Validación diferencial FALLÓ"
print("✓ Validación PASS")

## 2. Medición de performance (cuatro vías)

In [ ]:
bench_out = ROOT / "report" / "benchmark_nb.json"

sh([
    sys.executable, str(ROOT / "benchmarks" / "bench_four_way.py"),
    "--sizes", "4,8,16,32,48",
    "--repeats", "40",
    "--out", str(bench_out),
])

bench = json.loads(bench_out.read_text())

print("\n=== LATENCIA MEDIANA (µs) ===")
header = f"{'Size':>8}  {'pure_py':>10}  {'manual':>10}  {'generated':>10}  {'ipc':>10}  {'gen/manual':>10}"
print(header)
print("-" * len(header))

rows = []
for size, block in bench["implementations"].items():
    pure = block["pure_python"]["median"] * 1e6
    man = block["manual_ctypes"]["median"] * 1e6
    gen = block["generated"]["median"] * 1e6
    ipc = block["ipc"]["median"] * 1e6
    ratio = gen / man if man > 0 else float("nan")
    rows.append((int(size), pure, man, gen, ipc, ratio))
    print(f"{size + 'x' + size:>8}  {pure:10.1f}  {man:10.1f}  {gen:10.1f}  {ipc:10.1f}  {ratio:10.2f}x")

print("\n=== p99 (µs) ===")
print(f"{'Size':>8}  {'pure_py':>10}  {'manual':>10}  {'generated':>10}  {'ipc':>10}")
for size, block in bench["implementations"].items():
    print(
        f"{size + 'x' + size:>8}  "
        f"{block['pure_python']['p99']*1e6:10.1f}  "
        f"{block['manual_ctypes']['p99']*1e6:10.1f}  "
        f"{block['generated']['p99']*1e6:10.1f}  "
        f"{block['ipc']['p99']*1e6:10.1f}"
    )

## 3. Veredicto de eficiencia

In [ ]:
effort = bench.get("effort_proxy", {})
loc_gen = effort.get("generated", {}).get("loc_glue", 0)
loc_man = effort.get("manual_ctypes", {}).get("loc_glue", 25)

rows_ge8 = [r for r in rows if r[0] >= 8]
ratios = [r[5] for r in rows_ge8]
max_ratio = max(ratios) if ratios else float("inf")
median_ratio = statistics.median(ratios) if ratios else float("inf")

row32 = next((r for r in rows if r[0] == 32), rows[-1] if rows else None)
speedup_vs_pure = (row32[1] / row32[3]) if row32 and row32[3] > 0 else float("nan")

correct = diff["verdict"] == "PASS"
effort_ok = loc_gen < loc_man
perf_ok = max_ratio <= 1.25

print("=== CRITERIOS ===")
print(f"  Corrección (diferencial PASS)     : {'SÍ' if correct else 'NO'}  [{diff['verdict']}]")
print(f"  Esfuerzo (gen LOC < manual LOC)   : {'SÍ' if effort_ok else 'NO'}  [gen={loc_gen}, manual={loc_man}]")
print(f"  Performance (gen ≤ 1.25× manual)  : {'SÍ' if perf_ok else 'NO'}  [max={max_ratio:.2f}×, med={median_ratio:.2f}×]")
print(f"  Speedup vs pure Python (ref)      : {speedup_vs_pure:.1f}×")

if correct and effort_ok and perf_ok:
    verdict = "EFICIENTE"
    detail = "Adaptador correcto, menos esfuerzo que binding manual y rendimiento competitivo (≤ 1.25× manual)."
elif correct and effort_ok and not perf_ok:
    verdict = "CORRECTO PERO COSTOSO"
    detail = "Corrección y menor esfuerzo, pero latencia generada > 1.25× manual."
elif correct and not effort_ok:
    verdict = "CORRECTO SIN VENTAJA DE ESFUERZO"
    detail = "Correcto pero sin reducción de esfuerzo humano."
else:
    verdict = "NO APTO"
    detail = "Falló validación semántica o umbrales mínimos."

print("\n" + "=" * 60)
print(f"VEREDICTO DE EFICIENCIA: {verdict}")
print("=" * 60)
print(detail)
print("Decisión COMMONWEAVE:", "PROCEED" if verdict == "EFICIENTE" else "REVISAR / NO PROCEED")

## 4. (Opcional) Sello CSG

In [ ]:
sello_path = ROOT / "report" / "sello.json"
if sello_path.is_file():
    try:
        from seal.sello import SelloIntegridad
        data = json.loads(sello_path.read_text())
        vr = SelloIntegridad.verificar(data, base_dir=ROOT)
        print(json.dumps(vr, indent=2))
        print("✓ Sello CSG válido" if vr["signature_valid"] and vr["artifacts_intact"] else "✗ Sello inválido")
    except Exception as e:
        print("No se pudo verificar sello:", e)
else:
    print("Sin report/sello.json (opcional). Para generarlo: python run_zero.py")

## 5. Resumen exportable

In [ ]:
summary = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "root": str(ROOT),
    "validation": {
        "verdict": diff["verdict"],
        "pass_rate": diff["pass_rate"],
        "edge": f"{diff['edge_passed']}/{diff['edge_total']}",
    },
    "performance": {
        "sizes": [r[0] for r in rows],
        "median_us": {
            "pure_python": [round(r[1], 1) for r in rows],
            "manual_ctypes": [round(r[2], 1) for r in rows],
            "generated": [round(r[3], 1) for r in rows],
            "ipc": [round(r[4], 1) for r in rows],
        },
        "gen_over_manual_max": round(max_ratio, 3),
        "gen_over_manual_median": round(median_ratio, 3),
        "speedup_vs_pure_ref": round(speedup_vs_pure, 2),
    },
    "effort_loc_glue": {"generated": loc_gen, "manual_ctypes": loc_man},
    "efficiency_verdict": verdict,
    "decision": "PROCEED" if verdict == "EFICIENTE" else "REVISAR",
}

out_summary = ROOT / "report" / "veredicto_eficiencia.json"
out_summary.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(f"\nGuardado en {out_summary}")